In [1]:
# 02_robustness.ipynb
# ------------------------------------------------------------------------------------
# Imports, configuration, grayscale plotting style, and output savers.
# Design choices: single-cell notebook; English comments only; figures in grayscale via
# seaborn; no captions; dpi=600; every figure saved as BOTH .png and .pdf; table outputs
# saved as .csv; file paths are never printed (anonymity).
# ------------------------------------------------------------------------------------
import os, copy, hashlib, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# Reproducibility
GLOBAL_SEED = 2025
np.random.seed(GLOBAL_SEED)

# Paths (relative; intentionally never printed)
DATA_PATH   = os.path.join("..", "data", "criteo_clean.parquet")
FIG_DIR     = os.path.join("..", "results", "figures")
TBL_DIR     = os.path.join("..", "results", "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TBL_DIR, exist_ok=True)

# Grayscale, publication-style seaborn theme
sns.set_theme(context="paper", style="whitegrid")
GRAYSCALE = ["#111111", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYSCALE))
plt.rcParams.update({
    "image.cmap": "gray",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#000000",
    "text.color": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "font.size": 11,
    "savefig.bbox": "tight",
})

def save_fig(fig, name):
    """Save a figure as png and pdf at dpi=600. No captions, no path printing."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=600)
    plt.close(fig)

def save_table(df, name):
    """Save a results table as csv. No path printing."""
    df.to_csv(os.path.join(TBL_DIR, f"{name}.csv"), index=False)

# Experiment configuration
CFG = {
    "data": {
        "clean_path": DATA_PATH,
        # Only features observable at BIDDING TIME are used. The Criteo Attribution dataset's
        # click_pos/click_nb/cost/cpo/attribution are computed along the conversion PATH (e.g.
        # click_pos = "position of the click before a conversion", cpo = cost per ORDER), so they
        # are post-conversion and leak the label (each gives standalone AUROC ~1.0 vs conversion).
        # `click` is the stage-1 LABEL and must never be an input. We keep contextual categoricals,
        # the campaign id, and time_since_last_click (past-only recency).
        "cat_cols": ["campaign","cat1","cat2","cat3","cat4","cat5","cat6","cat7","cat8","cat9"],
        "dense_cols": ["time_since_last_click"],
        "stage_cols": ["click","conversion"],
        "max_vocab": 100000,
    },
    "split": {"use_existing_split": True, "train_max_day": 20, "valid_max_day": 24},
    "model": {"name":"leakfunnel","embed_dim":16,"hidden_dim":128,"n_heads":4,"n_layers":2,
              "dropout":0.1,"lambda_mono":0.1,"lambda_leak":1.0,"gate_lr_mult":50.0},
    # num_workers MUST be 0 on Windows + single-cell notebooks: spawned workers cannot
    # re-import classes defined in the notebook and will deadlock on the first batch.
    # We avoid DataLoader workers entirely by moving each split to the GPU up front
    # (see _tensors) and batching by index, so num_workers is not used for training.
    # amp (float16 mixed precision) is OFF by default: these models are small enough to be
    # fast in float32 on a consumer GPU, and AMP can produce NaN predictions on the
    # Transformer/ESMM paths. Turn it on only if you need the speed and have verified stability.
    "train": {"epochs":3,"batch_size":8192,"lr":1e-3,"weight_decay":1e-5,"num_workers":0,
              "device":"auto","seed":GLOBAL_SEED,"amp":False,"max_rows":None},
    # noise_sd=0.15 keeps the injected leak signal well above noise so EVERY leaky feature has a
    # strong, stable leak score (z_reg ~0.95), not just one by chance. leak_strength 0.5 with the
    # sparse 0/1 future_label then gives standalone AUROC ~0.98 on the training period.
    "leakage": {"as_of_days":[15,20,25],"leak_strength":0.5,"leak_fraction":0.5,
                "noise_sd":0.15,"family":"both"},
    "eval": {"n_seeds":5,"bootstrap":1000},
}

# FAST switch: set FAST = True for a quick pass (subsampled rows, fewer epochs/seeds) to
# iterate or sanity-check; set FAST = False for the full, paper-grade run. One line to flip.
FAST = False
if FAST:
    CFG["train"]["max_rows"] = 300000
    CFG["train"]["epochs"] = 2
    CFG["eval"]["n_seeds"] = 2

def audit_input_leakage(cfg, label_col="conversion", thresh=0.99, n=300000):
    # Self-check: measure each MODEL INPUT feature's standalone AUROC against the label on a
    # subsample. A value near 1.0 means the feature essentially encodes the label (a
    # post-outcome/leaky column that must not be an input). Prints a warning per offender.
    # This is a guard against silently re-introducing label leakage into the baseline pipeline.
    from sklearn.metrics import roc_auc_score
    cols = cfg["data"]["cat_cols"] + cfg["data"]["dense_cols"] + [label_col]
    t = pq.read_table(cfg["data"]["clean_path"], columns=list(dict.fromkeys(cols)))
    m = t.num_rows
    idx = np.linspace(0, m-1, min(n, m)).astype(np.int64)
    df = t.take(idx).to_pandas()
    y = df[label_col].to_numpy()
    flagged = []
    for c in cfg["data"]["cat_cols"] + cfg["data"]["dense_cols"]:
        x = np.nan_to_num(df[c].to_numpy().astype(float))
        try:
            a = roc_auc_score(y, x); a = max(a, 1 - a)
        except Exception:
            a = float("nan")
        if np.isfinite(a) and a >= thresh:
            flagged.append((c, a)); print(f"LEAKAGE WARNING: input '{c}' has standalone AUROC {a:.4f} vs {label_col} -> remove it")
    if not flagged:
        print(f"input leakage audit: OK (no input feature exceeds AUROC {thresh} vs {label_col})")
    return flagged

def report_device():
    # Print the compute device once so it is obvious whether CUDA is actually used.
    # If this shows CPU on a GPU machine, the installed torch is a CPU build -> reinstall
    # torch with the CUDA index URL (see README). No file paths are printed.
    try:
        import torch
        if torch.cuda.is_available():
            print(f"device: cuda ({torch.cuda.get_device_name(0)}) | torch {torch.__version__}")
        else:
            print(f"device: cpu | torch {torch.__version__} | CUDA not available")
    except ImportError:
        print("torch not installed (only data/detection notebooks run without it)")
# ------------------------------------------------------------------------------------
# Controlled leakage-injection harness (exact ground truth).
# row family: feature = PIT_prior + s * future_label + noise           -> detectable
# campaign family: per-campaign rate including post-as-of conversions   -> diluted/hidden
# ------------------------------------------------------------------------------------
def inject_features(cfg, seed=7, force_leaky=False, strengths=None, neutralize_live=False):
    # If neutralize_live=True, the leak signal (future_label / future-aware aggregate) is
    # REMOVED on the live period (test split): at live scoring time the future that leaked
    # into training does not exist yet, so a leaky feature collapses to its honest part
    # (prior + noise). This is what lets RQ3 measure robustness: a model that leaned on the
    # leak during training loses that signal live and degrades; a leakage-robust model does not.
    lk = cfg["leakage"]
    t = pq.read_table(cfg["data"]["clean_path"],
        columns=["campaign","timestamp","conversion","conversion_timestamp","day","split"]).to_pandas()
    # NOTE: inject over ALL rows (no max_rows truncation here). load_splits applies max_rows
    # per-split AFTER merging, keeping injected_df row-aligned with the clean frame.
    campaign=t["campaign"].to_numpy(); ts=t["timestamp"].to_numpy()
    conv=t["conversion"].to_numpy(); cts=t["conversion_timestamp"].to_numpy()
    is_live=(t["split"].to_numpy()=="test")   # the live period
    rng=np.random.default_rng(seed)
    NC=int(campaign.max())+1 if len(campaign) else 2
    future_label=((conv==1)&(cts>=0)).astype(np.float64)
    families=["row","campaign"] if lk["family"]=="both" else [lk["family"]]
    out=pd.DataFrame(index=t.index); gt={}
    for tau in lk["as_of_days"]:
        cutoff=tau*86400; imp=ts<=cutoff
        num=np.zeros(NC); den=np.zeros(NC)
        np.add.at(den, campaign[imp], 1.0)
        np.add.at(num, campaign[(conv==1)&(cts>=0)&(cts<=cutoff)], 1.0)
        prior=np.divide(num,den,out=np.zeros(NC),where=den>0)[campaign]
        for fam in families:
            is_leaky=True if force_leaky else (rng.random()<lk["leak_fraction"])
            s=(strengths[tau] if strengths else lk["leak_strength"]) if is_leaky else 0.0
            noise=rng.normal(0, lk["noise_sd"], size=len(conv))
            if fam=="row":
                # PURE label leak: no honest component. In training it strongly predicts the
                # label (s*future_label); when neutralized live it collapses to pure noise, so
                # a model that leaned on it has NOTHING left live. The honest predictive signal
                # lives elsewhere (the campaign embedding), which a robust model can rely on.
                sig = s*future_label
                if neutralize_live: sig = np.where(is_live, 0.0, sig)  # leak vanishes live
                feat = sig + noise
            else:
                if is_leaky:
                    fn=np.zeros(NC); fd=np.zeros(NC)
                    np.add.at(fd, campaign[imp], 1.0)
                    np.add.at(fn, campaign[(conv==1)&(cts>=0)&imp], 1.0)
                    fut=np.divide(fn,fd,out=np.zeros(NC),where=fd>0)[campaign]
                    feat=(1-s)*prior + s*fut + noise
                    if neutralize_live:  # live: fall back to honest prior (+noise)
                        feat = np.where(is_live, prior + noise, feat)
                else:
                    feat=prior+noise
            col=f"{fam}_feat_asof{tau}"
            out[col]=feat.astype(np.float32)
            gt[col]={"is_leaky":bool(is_leaky),"leak_strength":float(s),"family":fam,"as_of_day":int(tau)}
    out["conversion"]=conv; out["day"]=t["day"].to_numpy()
    return out, gt
# ------------------------------------------------------------------------------------
# Detection oracle (standalone AUROC vs as-of date) and baselines.
# ------------------------------------------------------------------------------------
def standalone_auroc(score, label):
    score=np.asarray(score,float); label=np.asarray(label)
    order=np.argsort(score,kind="mergesort"); r=np.empty(len(score)); r[order]=np.arange(1,len(score)+1)
    pos=label==1; npos=int(pos.sum()); nneg=int((~pos).sum())
    if npos==0 or nneg==0: return float("nan")
    return (r[pos].sum()-npos*(npos+1)/2)/(npos*nneg)

def auroc_vs_date(feature, label, day, as_of_days):
    out={}
    for tau in as_of_days:
        m=day<=tau
        if m.sum()<10: out[tau]=float("nan"); continue
        a=standalone_auroc(feature[m], label[m]); out[tau]=max(a,1-a)
    return out

def leak_score_from_curve(curve):
    # DETECTION score (RQ1): a leak shows as discriminative power RISING across as-of dates.
    # peak-above-0.5 plus positive slope separates leaky (rising) from clean (flat) features.
    vals=np.array([curve[t] for t in sorted(curve)],float); vals=vals[~np.isnan(vals)]
    if len(vals)==0: return 0.0
    peak=max(0.0, vals.max()-0.5); slope=max(0.0, vals[-1]-vals[0]) if len(vals)>=2 else 0.0
    return float(peak+slope)

def score_all_features(df, feature_cols, as_of_days, label_col="conversion", day_col="day"):
    label=df[label_col].to_numpy(); day=df[day_col].to_numpy(); res={}
    for c in feature_cols:
        curve=auroc_vs_date(df[c].to_numpy(), label, day, as_of_days)
        res[c]={"curve":curve,"z":leak_score_from_curve(curve)}
    return res

def regularizer_scores(df, feature_cols, label_col="conversion"):
    # REGULARIZER score z_reg in [0,1]: how strongly a feature discriminates the label overall,
    # = 2*(standalone_AUROC - 0.5). This is the magnitude the leak gate must shrink. Distinct
    # from the detection score above (which keys on the date-rising signature, not magnitude).
    label=df[label_col].to_numpy(); res={}
    for c in feature_cols:
        a=standalone_auroc(df[c].to_numpy(), label)
        a=max(a,1-a) if np.isfinite(a) else 0.5
        res[c]=float(max(0.0, 2.0*(a-0.5)))
    return res

def correlation_shift_scores(df, feature_cols, as_of_days, label_col="conversion", day_col="day"):
    label=df[label_col].to_numpy().astype(float); day=df[day_col].to_numpy(); out={}
    for c in feature_cols:
        f=df[c].to_numpy().astype(float); corrs=[]
        for tau in as_of_days:
            m=day<=tau
            if m.sum()<10 or f[m].std()==0: continue
            corrs.append(abs(np.corrcoef(f[m],label[m])[0,1]))
        out[c]=float(max(corrs)-min(corrs)) if len(corrs)>=2 else 0.0
    return out

def evaluate_detection(scores, gt, score_key="z"):
    from sklearn.metrics import average_precision_score, roc_auc_score
    feats=[f for f in scores if f in gt]
    y=np.array([1 if gt[f]["is_leaky"] else 0 for f in feats])
    s=np.array([scores[f][score_key] if isinstance(scores[f],dict) else scores[f] for f in feats])
    m={"n_features":len(feats),"n_leaky":int(y.sum())}
    if y.sum() in (0,len(y)): m["auprc"]=float("nan"); m["auroc"]=float("nan"); m["precision"]=float("nan"); m["recall"]=float("nan"); return m
    m["auprc"]=float(average_precision_score(y,s)); m["auroc"]=float(roc_auc_score(y,s))
    thr=(s.max()+s.min())/2 if s.max()>s.min() else s.mean(); pred=(s>=thr).astype(int)
    tp=int(((pred==1)&(y==1)).sum()); fp=int(((pred==1)&(y==0)).sum()); fn=int(((pred==0)&(y==1)).sum())
    m["precision"]=tp/(tp+fp) if tp+fp else 0.0; m["recall"]=tp/(tp+fn) if tp+fn else 0.0
    return m
# ------------------------------------------------------------------------------------
# Models: LeakFunnel (proposed), AITM, ESMM, MLP; training and evaluation utilities.
# ------------------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from sklearn.metrics import roc_auc_score, average_precision_score

class FeatureEmbedding(nn.Module):
    # Vectorized feature tokenizer. Categoricals share ONE padded embedding table with
    # per-field id offsets (a single gather instead of a Python loop over fields); dense
    # features become tokens via one broadcasted multiply. Output: [B, n_cat+n_dense, D].
    def __init__(self, vocab_sizes, n_dense, d):
        super().__init__()
        self.n_cat=len(vocab_sizes); self.n_dense=n_dense; self.d=d
        if self.n_cat>0:
            offs=np.concatenate([[0], np.cumsum(vocab_sizes)[:-1]]).astype(np.int64)
            self.register_buffer("offsets", torch.tensor(offs, dtype=torch.long))
            self.cat_emb=nn.Embedding(int(sum(vocab_sizes)), d)
        if n_dense>0:
            self.dense_proj=nn.Parameter(torch.randn(n_dense,d)*0.01)
            self.dense_bias=nn.Parameter(torch.zeros(n_dense,d))
    def forward(self, cat, dense):
        parts=[]
        if self.n_cat>0:
            parts.append(self.cat_emb(cat.long() + self.offsets))    # [B, n_cat, D] one gather
        if self.n_dense>0:
            parts.append(dense.unsqueeze(-1)*self.dense_proj + self.dense_bias)  # [B, n_dense, D]
        return torch.cat(parts, dim=1) if len(parts)>1 else parts[0]

class LeakFunnel(nn.Module):
    # Leakage robustness is enforced at the INPUT, not via attention. Each feature token has a
    # learnable gate g_j = sigmoid(theta_j) in [0,1] applied to its embedding BEFORE the encoder,
    # so a gated-out feature cannot reach the representation at all (no "back door" through the
    # encoder's self-attention). The leak regularizer pushes gates of high-leak-score features
    # toward 0: penalty = sum_j z_j * g_j. The detector's per-feature score z_j (nonzero only on
    # injected/leaky features) thus directly shrinks the model's use of leaky inputs.
    def __init__(self, vocab_sizes, n_dense, n_stages, cfg, injected_mask=None):
        super().__init__(); d=cfg["model"]["embed_dim"]; h=cfg["model"]["hidden_dim"]
        self.embed=FeatureEmbedding(vocab_sizes,n_dense,d)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=cfg["model"]["n_heads"],dim_feedforward=h,
                                       dropout=cfg["model"]["dropout"],batch_first=True)
        self.encoder=nn.TransformerEncoder(enc,num_layers=cfg["model"]["n_layers"])
        self.n_stages=n_stages; self.n_cat=len(vocab_sizes); self.n_dense=n_dense
        self.n_tokens=self.n_cat+n_dense
        # one gate parameter per input token (categoricals first, then dense). Init at logit 0
        # (gate=0.5): the sigmoid slope is maximal there (0.25), so the leak regularizer can
        # actually push a leaky feature's gate toward 0. (Init at a high logit like 3.0 has
        # slope ~0.05 and barely moves -> the gate never closes.)
        self.gate_logit=nn.Parameter(torch.zeros(self.n_tokens))
        self.stage_queries=nn.Parameter(torch.randn(n_stages,d)*0.02)
        self.stage_heads=nn.ModuleList([nn.Linear(d,1) for _ in range(n_stages)]); self.scale=d**0.5
        mask=injected_mask if injected_mask is not None else torch.zeros(n_dense)
        self.register_buffer("injected_mask", mask.float())   # kept for compatibility
    def gates(self):
        return torch.sigmoid(self.gate_logit)                 # [n_tokens] in (0,1)
    def forward(self, cat, dense, return_attn=False):
        tok=self.embed(cat,dense)                             # [B, T, D]
        g=self.gates().view(1,-1,1)                           # [1, T, 1]
        enc=self.encoder(tok*g); B,T,D=enc.shape              # gate BEFORE the encoder
        q=self.stage_queries.unsqueeze(0).expand(B,-1,-1)
        attn=F.softmax(torch.matmul(q,enc.transpose(1,2))/self.scale, dim=-1)
        ctx=torch.matmul(attn,enc)
        logits=torch.stack([self.stage_heads[s](ctx[:,s,:]).squeeze(-1) for s in range(self.n_stages)],dim=1)
        probs=torch.sigmoid(logits)
        return (probs,logits,attn) if return_attn else (probs,logits)
    def leak_penalty(self, leak_scores_tokens):
        # leak_scores_tokens: [n_tokens] per-token leak score z (nonzero on leaky features).
        # Penalize the OPEN gate mass on leaky tokens so the model closes them.
        g=self.gates()
        w=leak_scores_tokens.to(g.device)
        return (w*g).sum()

class AITM(nn.Module):
    def __init__(self, vocab_sizes, n_dense, n_stages, cfg, injected_mask=None):
        super().__init__(); d=cfg["model"]["embed_dim"]; h=cfg["model"]["hidden_dim"]
        self.embed=FeatureEmbedding(vocab_sizes,n_dense,d); in_dim=(len(vocab_sizes)+n_dense)*d
        self.towers=nn.ModuleList([nn.Sequential(nn.Linear(in_dim,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU()) for _ in range(n_stages)])
        self.transfer=nn.ModuleList([nn.Linear(h,h) for _ in range(n_stages)])
        self.heads=nn.ModuleList([nn.Linear(h,1) for _ in range(n_stages)]); self.n_stages=n_stages
    def forward(self, cat, dense, return_attn=False):
        x=self.embed(cat,dense).flatten(1); reps=[t(x) for t in self.towers]; logits=[]; prev=None
        for s in range(self.n_stages):
            r=reps[s]
            if prev is not None: r=r+torch.sigmoid(self.transfer[s](prev))*prev
            logits.append(self.heads[s](r).squeeze(-1)); prev=r
        logits=torch.stack(logits,dim=1); probs=torch.sigmoid(logits)
        return (probs,logits,None) if return_attn else (probs,logits)

class ESMM(nn.Module):
    def __init__(self, vocab_sizes, n_dense, n_stages, cfg, injected_mask=None):
        super().__init__(); assert n_stages==2
        d=cfg["model"]["embed_dim"]; h=cfg["model"]["hidden_dim"]
        self.embed=FeatureEmbedding(vocab_sizes,n_dense,d); in_dim=(len(vocab_sizes)+n_dense)*d
        self.ctr=nn.Sequential(nn.Linear(in_dim,h),nn.ReLU(),nn.Linear(h,1))
        self.cvr=nn.Sequential(nn.Linear(in_dim,h),nn.ReLU(),nn.Linear(h,1))
    def forward(self, cat, dense, return_attn=False):
        x=self.embed(cat,dense).flatten(1)
        # compute in float32 so the probability product and its logit inverse stay finite
        # under AMP (float16 would overflow log(p/(1-p)) when p is near 0 or 1 -> NaN).
        p_ctr=torch.sigmoid(self.ctr(x).squeeze(-1).float())
        p_cvr=torch.sigmoid(self.cvr(x).squeeze(-1).float())
        probs=torch.stack([p_ctr, p_ctr*p_cvr],dim=1)
        pc=probs.clamp(1e-6,1-1e-6); logits=torch.log(pc/(1-pc))
        return (probs,logits,None) if return_attn else (probs,logits)

class MLP(nn.Module):
    def __init__(self, vocab_sizes, n_dense, n_stages, cfg, injected_mask=None):
        super().__init__(); d=cfg["model"]["embed_dim"]; h=cfg["model"]["hidden_dim"]
        self.embed=FeatureEmbedding(vocab_sizes,n_dense,d); in_dim=(len(vocab_sizes)+n_dense)*d
        self.bottom=nn.Sequential(nn.Linear(in_dim,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU())
        self.heads=nn.ModuleList([nn.Linear(h,1) for _ in range(n_stages)]); self.n_stages=n_stages
    def forward(self, cat, dense, return_attn=False):
        r=self.bottom(self.embed(cat,dense).flatten(1))
        logits=torch.stack([hd(r).squeeze(-1) for hd in self.heads],dim=1); probs=torch.sigmoid(logits)
        return (probs,logits,None) if return_attn else (probs,logits)

def build_model(name, vocab_sizes, n_dense, n_stages, cfg, injected_mask=None):
    return {"leakfunnel":LeakFunnel,"aitm":AITM,"esmm":ESMM,"mlp":MLP}[name.lower()](
        vocab_sizes,n_dense,n_stages,cfg,injected_mask=injected_mask)

def stage_monotonicity_penalty(probs):
    if probs.shape[1]<2: return probs.new_zeros(())
    return F.relu(probs[:,1:]-probs[:,:-1]).mean()

def _hash_col(v, mod):
    v=v.astype(np.int64); return ((np.abs(v)*np.int64(2654435761))%np.int64(mod)).astype(np.int64)

class CriteoFunnel(Dataset):
    def __init__(self, df, cfg, stats=None, injected_cols=None):
        cat_cols=cfg["data"]["cat_cols"]; dense_cols=cfg["data"]["dense_cols"]
        stage_cols=cfg["data"]["stage_cols"]; mv=cfg["data"]["max_vocab"]; injected_cols=injected_cols or []
        arrs=[]; self.vocab_sizes=[]
        for c in cat_cols:
            size=max(2, min(mv, int(df[c].max())+2) if len(df) else 2)
            arrs.append(_hash_col(df[c].to_numpy(), size)); self.vocab_sizes.append(size)
        self.cat=np.stack(arrs,1).astype(np.int64) if arrs else np.zeros((len(df),0),np.int64)
        dcols=list(dense_cols)+list(injected_cols)
        dense=df[dcols].to_numpy().astype(np.float32) if dcols else np.zeros((len(df),0),np.float32)
        if stats is None:
            mean=dense.mean(0) if dense.shape[1] else np.zeros(0,np.float32)
            std=dense.std(0)+1e-6 if dense.shape[1] else np.ones(0,np.float32)
            self.stats={"mean":mean,"std":std}
        else: self.stats=stats
        if dense.shape[1]: dense=(dense-self.stats["mean"])/self.stats["std"]
        self.dense=dense; self.n_injected=len(injected_cols); self.injected_cols=list(injected_cols)
        self.y=df[stage_cols].to_numpy().astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return torch.from_numpy(self.cat[i]), torch.from_numpy(self.dense[i]), torch.from_numpy(self.y[i])

def load_splits(cfg, injected_df=None):
    cols=list(dict.fromkeys(cfg["data"]["cat_cols"]+cfg["data"]["dense_cols"]+cfg["data"]["stage_cols"]+["day","split"]))
    df=pq.read_table(cfg["data"]["clean_path"], columns=cols).to_pandas()
    injected_cols=[]
    if injected_df is not None:
        injected_cols=[c for c in injected_df.columns if "_feat_asof" in c]
        for c in injected_cols: df[c]=injected_df[c].to_numpy()
    if cfg["split"]["use_existing_split"] and "split" in df.columns:
        tr=df[df.split=="train"].reset_index(drop=True); va=df[df.split=="valid"].reset_index(drop=True); te=df[df.split=="test"].reset_index(drop=True)
    else:
        tmd=cfg["split"]["train_max_day"]; vmd=cfg["split"]["valid_max_day"]
        tr=df[df.day<=tmd].reset_index(drop=True); va=df[(df.day>tmd)&(df.day<=vmd)].reset_index(drop=True); te=df[df.day>vmd].reset_index(drop=True)
    # max_rows subsamples WITHIN each split (proportional) so FAST mode never empties a split.
    mr=cfg["train"].get("max_rows")
    if mr:
        tot=len(tr)+len(va)+len(te)
        if tot>0:
            frac=int(mr)/tot
            tr=tr.iloc[:max(1,int(len(tr)*frac))].reset_index(drop=True)
            va=va.iloc[:max(1,int(len(va)*frac))].reset_index(drop=True)
            te=te.iloc[:max(1,int(len(te)*frac))].reset_index(drop=True)
    train_ds=CriteoFunnel(tr,cfg,None,injected_cols)
    valid_ds=CriteoFunnel(va,cfg,train_ds.stats,injected_cols)
    test_ds=CriteoFunnel(te,cfg,train_ds.stats,injected_cols)
    meta={"vocab_sizes":train_ds.vocab_sizes,"n_dense":train_ds.dense.shape[1],
          "n_cat":train_ds.cat.shape[1],"n_stages":len(cfg["data"]["stage_cols"]),
          "injected_cols":injected_cols,"n_injected":len(injected_cols)}
    return train_ds, valid_ds, test_ds, meta

def resolve_device(pref):
    if pref and pref!="auto": return pref
    if torch.cuda.is_available(): return "cuda"
    if getattr(torch.backends,"mps",None) and torch.backends.mps.is_available(): return "mps"
    return "cpu"

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def _tensors(ds, device):
    # Move an entire split to the device ONCE as tensors. No DataLoader, no worker
    # processes -> cannot deadlock on Windows, and no per-batch host->device copies.
    # int32 ids keep embeddings valid while halving memory vs int64. Falls back to
    # CPU-resident tensors (indexing still works, just slower) if the GPU is out of memory.
    cat=torch.from_numpy(ds.cat.astype(np.int32))
    dense=torch.from_numpy(ds.dense)   # already float32
    y=torch.from_numpy(ds.y)           # float32
    try:
        return cat.to(device), dense.to(device), y.to(device)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            print("note: split did not fit on GPU; keeping it in CPU memory (slower).")
            return cat, dense, y
        raise

@torch.no_grad()
def evaluate(model, ds, cfg, meta, device):
    model.eval()
    cat_all, dense_all, y_all = _tensors(ds, device)
    bs=cfg["train"]["batch_size"]*4
    n=cat_all.shape[0]
    P=[[] for _ in range(meta["n_stages"])]
    for i in range(0, n, bs):
        cb=cat_all[i:i+bs].to(device, non_blocking=True); db=dense_all[i:i+bs].to(device, non_blocking=True)
        probs,_=model(cb, db)
        probs=probs.float().cpu().numpy()
        for s in range(meta["n_stages"]): P[s].append(probs[:,s])
    y_np=y_all.cpu().numpy()
    out={}; names=cfg["data"]["stage_cols"]
    for s in range(meta["n_stages"]):
        p=np.concatenate(P[s]); yy=y_np[:,s]; sn=names[s]
        # Guard against non-finite predictions (can arise from AMP instability): replace
        # with the neutral 0.5 and warn, so scoring does not crash and the issue is visible.
        if not np.all(np.isfinite(p)):
            n_bad=int((~np.isfinite(p)).sum())
            print(f"warning: {n_bad} non-finite predictions at stage '{sn}' set to 0.5")
            p=np.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)
        if yy.min()==yy.max(): out[f"{sn}_auroc"]=float("nan"); out[f"{sn}_prauc"]=float("nan")
        else: out[f"{sn}_auroc"]=float(roc_auc_score(yy,p)); out[f"{sn}_prauc"]=float(average_precision_score(yy,p))
    return out

def train_one(cfg, train_ds, valid_ds, test_ds, meta, leak_scores=None, model_name=None, seed=0):
    device=resolve_device(cfg["train"]["device"]); set_seed(seed); name=model_name or cfg["model"]["name"]
    if device=="cuda": torch.backends.cudnn.benchmark=True
    n_dense=meta["n_dense"]; injected_mask=torch.zeros(n_dense)
    if meta["n_injected"]>0: injected_mask[-meta["n_injected"]:]=1.0
    model=build_model(name, meta["vocab_sizes"], n_dense, meta["n_stages"], cfg, injected_mask).to(device)
    # Put the feature gates (LeakFunnel.gate_logit) in their own param group: a much higher
    # learning rate and NO weight decay, so the leak regularizer can actually drive leaky gates
    # to 0 within a few epochs. The rest of the network uses the normal lr / weight decay.
    gate_lr = cfg["train"]["lr"] * cfg["model"].get("gate_lr_mult", 50.0)
    gate_params=[p for n_,p in model.named_parameters() if n_.endswith("gate_logit")]
    other_params=[p for n_,p in model.named_parameters() if not n_.endswith("gate_logit")]
    groups=[{"params":other_params,"lr":cfg["train"]["lr"],"weight_decay":cfg["train"]["weight_decay"]}]
    if gate_params:
        groups.append({"params":gate_params,"lr":gate_lr,"weight_decay":0.0})
    opt=torch.optim.Adam(groups)
    use_amp=bool(cfg["train"]["amp"]) and device=="cuda"; scaler=torch.amp.GradScaler("cuda", enabled=use_amp)
    # Per-TOKEN leak scores: tokens are [categoricals (n_cat)] then [dense (n_dense)]; injected
    # leaky features occupy the tail of the dense block. z is nonzero only there.
    n_cat=meta["n_cat"]; n_tokens=n_cat+n_dense
    z=torch.zeros(n_tokens, device=device)
    if leak_scores and meta["n_injected"]>0:
        base=n_cat + (n_dense - meta["n_injected"])
        for i,col in enumerate(meta["injected_cols"]): z[base+i]=float(leak_scores.get(col,0.0))
    lam_m=cfg["model"]["lambda_mono"]; lam_l=cfg["model"]["lambda_leak"]
    # Whole training split resident on the device; shuffle by index each epoch.
    cat_all, dense_all, y_all = _tensors(train_ds, device)
    n=cat_all.shape[0]; bs=cfg["train"]["batch_size"]
    for ep in range(cfg["train"]["epochs"]):
        model.train()
        perm=torch.randperm(n, device=cat_all.device)
        for i in range(0, n, bs):
            idx=perm[i:i+bs].to(cat_all.device)
            cat=cat_all[idx].to(device, non_blocking=True)
            dense=dense_all[idx].to(device, non_blocking=True)
            y=y_all[idx].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                if name=="leakfunnel": probs,logits,attn=model(cat,dense,return_attn=True)
                else: probs,logits=model(cat,dense); attn=None
                # BCE on clamped probabilities in float32: stable under AMP (avoids
                # logit overflow -> NaN). Equivalent objective, finite everywhere.
                p=probs.float().clamp(1e-6, 1-1e-6)
                loss=sum(F.binary_cross_entropy(p[:,s], y[:,s]) for s in range(meta["n_stages"]))
                if name=="leakfunnel":
                    loss=loss+lam_m*stage_monotonicity_penalty(probs)
                    if z.abs().sum()>0: loss=loss+lam_l*model.leak_penalty(z)
            # Guard: skip a batch whose loss went non-finite so it cannot poison the weights.
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return evaluate(model, test_ds, cfg, meta, device)

# ------------------------------------------------------------------------------------
# RQ3: robustness WITH leakage; RQ4: parity with NO leakage.
# Models: LeakFunnel (proposed), AITM, ESMM, MLP. >=5 seeds per CFG.
# Tables to results/tables; grayscale bar chart to results/figures.
# ------------------------------------------------------------------------------------
# IMPORTANT: report the final RQ3/RQ5 numbers with FAST=False (full data, >=3 epochs, GPU).
# FAST mode subsamples rows and makes the leak-gate training unstable, which blurs the
# full-vs-no_leakreg gap. Use FAST only to check the pipeline runs.
report_device()          # confirm CUDA is actually used; CPU here means a CPU-only torch build
audit_input_leakage(CFG)  # self-check: no model INPUT feature may encode the label

def run_robustness(cfg, with_leak):
    model_names = ["leakfunnel", "aitm", "esmm", "mlp"]
    seeds = list(range(cfg["eval"]["n_seeds"]))
    injected_df = None; leak_scores = None
    if with_leak:
        c = copy.deepcopy(cfg); c["leakage"]["family"] = "row"
        # Detector sees the TRAINING-time features (leak present) to score leakiness...
        detect_df, gt = inject_features(c, seed=7)
        # leak_scores drive the model's leak regularizer: magnitude of label discrimination.
        leak_scores = regularizer_scores(detect_df, list(gt.keys()))
        # ...but models train/evaluate on features whose leak is NEUTRALIZED in the live
        # (test) period, so robustness = how well a model holds up when the leak vanishes live.
        injected_df, _ = inject_features(c, seed=7, neutralize_live=True)
    train_ds, valid_ds, test_ds, meta = load_splits(cfg, injected_df=injected_df)
    rows = []
    for name in model_names:
        for seed in seeds:
            r = train_one(cfg, train_ds, valid_ds, test_ds, meta, leak_scores, name, seed)
            r.update({"model": name, "seed": seed, "with_leak": with_leak})
            rows.append(r)
    return pd.DataFrame(rows)

rq3 = run_robustness(CFG, with_leak=True);  save_table(rq3, "rq3_robustness_withleak")
rq4 = run_robustness(CFG, with_leak=False); save_table(rq4, "rq4_noleak_parity")

agg3 = rq3.groupby("model")["conversion_auroc"].agg(["mean", "std"]).reset_index()
order = ["leakfunnel", "aitm", "esmm", "mlp"]
agg3 = agg3.set_index("model").loc[order].reset_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(agg3["model"], agg3["mean"], yerr=agg3["std"], capsize=4,
       color=["#111111", "#666666", "#999999", "#cccccc"], edgecolor="#000000")
ax.set_ylabel("conversion AUROC (test, leakage present)")
ax.set_xlabel("model")
save_fig(fig, "rq3_robustness_auroc")

# combined mean table across leak/no-leak for the paper
combined = (pd.concat([rq3, rq4])
            .groupby(["model", "with_leak"])["conversion_auroc"]
            .mean().reset_index())
save_table(combined, "rq3_rq4_conversion_auroc_means")
combined


device: cuda (NVIDIA GeForce RTX 4060 Ti) | torch 2.6.0+cu124
input leakage audit: OK (no input feature exceeds AUROC 0.99 vs conversion)


,model,with_leak,conversion_auroc
0,aitm,False,0.842238
1,aitm,True,0.601773
2,esmm,False,0.842561
3,esmm,True,0.616884
4,leakfunnel,False,0.840775
5,leakfunnel,True,0.640728
6,mlp,False,0.842371
7,mlp,True,0.589234
